### First version of the Spherinator preprocessing routine to produce point clouds or 2D maps of arbitrary quantities 

In [1]:
import requests
import numpy as np
import matplotlib.pyplot as plt
import pynbody
from PIL import Image, ImageFilter
from pathlib import Path
from scipy.stats import binned_statistic_2d

In [2]:
### common functions


def rotate_galaxy(particles, orientation, spin_aperture):  # [kpc]
    if orientation == "original":
        return particles

    elif orientation in ["face-on", "edge-on"]:
        rad = np.linalg.norm(particles["Coordinates"])
        inner_mask = rad < spin_aperture
        print(f"  particles within spin aperture: {sum(inner_mask)}")
        pos_inner = particles["Coordinates"][inner_mask][:, 0:3]
        vel_inner = particles["Velocities"][inner_mask][:, 0:3]
        mass_inner = particles["Masses"][inner_mask]
        sL = np.cross(pos_inner, vel_inner)
        Lvec = np.sum(mass_inner[:, np.newaxis] * sL, axis=0)
        spin = Lvec / np.linalg.norm(Lvec)
        # print('  angular momentum:', Lvec)
        print("  spin (unit) vector:", spin)

    if orientation == "random":
        # random vector as spin
        spin = np.random.normal(loc=0, scale=1, size=3)
        spin = spin / np.linalg.norm(spin)
        print("  using random orientation vector:", spin)

    pos = particles["Coordinates"][:, 0:3]
    vel = particles["Velocities"][:, 0:3]

    pos_rot = rotate_z(pos, np.arctan2(spin[0], spin[1]) * 180.0 / np.pi)
    vel_rot = rotate_z(vel, np.arctan2(spin[0], spin[1]) * 180.0 / np.pi)
    norm = rotate_z(np.array([spin]), np.arctan2(spin[0], spin[1]) * 180.0 / np.pi)[0]

    pos_rot = rotate_x(pos_rot, np.arctan2(norm[1], norm[2]) * 180.0 / np.pi)
    vel_rot = rotate_x(vel_rot, np.arctan2(norm[1], norm[2]) * 180.0 / np.pi)
    norm = rotate_x(np.array([norm]), np.arctan2(norm[1], norm[2]) * 180.0 / np.pi)[0]

    print(f"  spin in new rotated frame (should be [0,0,1]): {norm[0]:.3f},{norm[1]:.3f},{norm[2]:.3f}")

    particles["Coordinates"] = pos_rot
    particles["Velocities"] = vel_rot

    return particles


def create_image(
    particles,
    field,
    operation,
    fov,
    image_depth,
    image_scale,
    image_size,
    smoothing,
    subid,
    component,
    orientation,
    output_path,
    debug,
):

    if type(fov) == float:
        scale = fov
    else:
        if fov == "scaled":
            rad = np.linalg.norm(particles["Coordinates"], axis=1)
            if debug:
                print(np.min(particles["Coordinates"][:, 0]), np.max(particles["Coordinates"][:, 0]))
                print(np.min(particles["Coordinates"][:, 1]), np.max(particles["Coordinates"][:, 1]))
                print(np.min(particles["Coordinates"][:, 2]), np.max(particles["Coordinates"][:, 2]))
            max_rad = 1.2 * np.percentile(rad, 99)
            print(f" min, median, max radius: {np.min(rad):.1f},{np.median(rad):.1f},{np.max(rad):.1f} kpc")

    print(f" FOV: {2 * max_rad:.1f} kpc")

    if orientation in ["face-on", "original", "random"]:
        indy = 1

    elif orientation == "edge-on":
        indy = 2

    img_x = particles["Coordinates"][:, 0]
    img_y = particles["Coordinates"][:, indy]
    if field == "HI mass":
        quantity = particles["Masses"] * particles["NeutralHydrogenAbundance"]
    else:
        quantity = particles[field]

    # define image resolution and physical extent
    nPixels = [image_size, image_size]
    minMax = [-max_rad, max_rad]  # [kpc], relative to the galaxy center
    pixelScale = 2 * max_rad / float(image_size)

    # add the mass of particles on a grid in the image plane
    # print(len(particles['Masses']), len(img_x), len(img_y))
    grid_quantity, _, _, _ = binned_statistic_2d(
        img_x, img_y, quantity, statistic=operation, bins=nPixels, range=[minMax, minMax]
    )
    # count the number of particles on the grid
    grid_npart, _, _, _ = binned_statistic_2d(
        img_x, img_y, quantity, statistic="count", bins=nPixels, range=[minMax, minMax]
    )

    # make image and save
    part_mass = np.mean(particles["Masses"])
    print(f" mean particle mass = {part_mass:.2e} Ms")
    image = grid_quantity
    image = np.clip(image, image_depth * part_mass, np.inf)
    print(f" grid values: min={np.min(image.flatten()):.2e} Ms, max={np.max(image.flatten()):.2e} Ms")
    if image_scale == "log":
        image = np.log10(image)

    if np.max(image) > np.min(image):
        image = (image - np.min(image)) / (np.max(image) - np.min(image))
    else:
        image = np.zeros_like(image)

    # Plot histogram
    if debug:
        plt.figure()
        plt.hist(image.flatten(), bins=100, color="gray", alpha=0.7)
        plt.title("Histogram of Grid Values")
        plt.xlabel("Intensity")
        plt.ylabel("Frequency")

    image = Image.fromarray((np.clip(image, 0, 1) * 255).astype(np.uint8), mode="L")
    if smoothing > 0:
        image = image.filter(ImageFilter.GaussianBlur(radius=smoothing / pixelScale))

    # filepath = output_path / Path(sim, str(snapshot))
    filepath = Path(output_path)
    filepath.mkdir(parents=True, exist_ok=True)
    filename = filepath / Path(str(subid) + "_" + component + "_" + field + ".jpg")
    image.save(filename)

    return


def rotate_x(ar, angle):
    """Rotates the snapshot about the current x-axis by 'angle' degrees."""
    angle *= np.pi / 180
    mat = np.matrix([[1, 0, 0], [0, np.cos(angle), -np.sin(angle)], [0, np.sin(angle), np.cos(angle)]])
    return np.array(np.dot(mat, ar.transpose()).transpose())


def rotate_y(ar, angle):
    """Rotates the snapshot about the current y-axis by 'angle' degrees."""
    angle *= np.pi / 180
    mat = np.matrix([[np.cos(angle), 0, np.sin(angle)], [0, 1, 0], [-np.sin(angle), 0, np.cos(angle)]])
    return np.array(np.dot(mat, ar.transpose()).transpose())


def rotate_z(ar, angle):
    """Rotates the snapshot about the current z-axis by 'angle' degrees."""
    angle *= np.pi / 180
    mat = np.matrix([[np.cos(angle), -np.sin(angle), 0], [np.sin(angle), np.cos(angle), 0], [0, 0, 1]])
    return np.array(np.dot(mat, ar.transpose()).transpose())


### API helper function


def get(path, params=None):
    # make HTTP GET request to path
    headers = {"api-key": "5a21bb189d49e865c26249c8aad50c2f"}
    r = requests.get(path, params=params, headers=headers)

    # raise exception if response code is not HTTP SUCCESS (200)
    r.raise_for_status()

    if r.headers["content-type"] == "application/json":
        return r.json()  # parse json responses automatically

    if "content-disposition" in r.headers:
        filename = r.headers["content-disposition"].split("filename=")[1]
        with open(filename, "wb") as f:
            f.write(r.content)
        return filename  # return the filename string

    return r

In [3]:
def data_preprocess_local_pynbody(
    snapshot_path,
    halos_path=None,
    objects="centrals",
    selection_type="stellar mass",
    min_mass=1e8,
    max_mass=np.inf,
    component="stars",
    output_type="2D projection",
    field="mass",
    operation="sum",
    fov="scaled",  # [kpc]
    image_depth=1,  # [particles]
    image_size=128,
    smoothing=1.0,  # [kpc]
    channels=1,
    image_scale="log",
    orientation="face-on",
    spin_aperture=15.0,  # [kpc]
    catalog_fields=["SubhaloStarMetallicity", "SubhaloSFR"],
    resolution_limit=1e9,  # [Msun]
    output_path="./images/",
    debug=False,
):

    print(
        f"Parameters:\n"
        f" snapshot_path: {snapshot_path}\n"
        f" halos_path: {halos_path}\n"
        f" objects: {objects}\n"
        f" selection_type: {selection_type}\n"
        f" min_mass: {min_mass:.2e} Ms, max_mass: {max_mass:.2e} Ms\n"
        f" component: {component}\n"
        f" output_type: {output_type}\n"
        f" field: {field}\n"
        f" operation: {operation}\n"
        f" fov: {fov}\n"
        f" image_depth: {image_depth} particles\n"
        f" image_size: {image_size}\n"
        f" smoothing: {smoothing} kpc\n"
        f" channels: {channels}\n"
        f" image_scale: {image_scale}\n"
        f" orientation: {orientation}\n"
        f" spin_aperture: {spin_aperture:.1f} kpc\n"
        f" catalog_fields: {catalog_fields}\n"
        f" resolution_limit: {resolution_limit:.2e} Ms\n"
        f" output_path: {output_path}\n"
    )

    # load snapshot
    print(f"loading simulation snapshot from {snapshot_path}...")
    f = pynbody.load(snapshot_path)

    print("\n snapshot properties:\n", f.properties)
    print("\n particles families:\n", f.families())
    print("\n loadable keys:\n", f.loadable_keys())

    print("\n Whole snapshot Ngas = %e, Ndark = %e, Nstar = %e\n" % (len(f.gas), len(f.dark), len(f.star)))

    # convert to physical units
    f.physical_units()

    # load halo catalog
    if halos_path is not None:
        print(f"loading halo catalog from {halos_path}...")
        h = f.halos(filename=halos_path, subhalos=True)  # includes all subhalos
    else:
        print("loading halo catalog from sanpshot...")
        h = f.halos(subhalos=True)
    print(h)

    print("\n Number of halos:", len(h))
    print("\n Number of particles in the first halo:", len(h[0]))

    print(h[0].properties)

    # convert to physical units
    h.physical_units()

    # load all halo data into memory
    h.load_all()

    # Loop over galaxies to read galaxy properties and particle data
    sub_id = []
    # group_id = []
    m_stellar = []
    m_halo = []
    m_tot = []
    r_half = []
    var0 = []
    var1 = []
    print(f"selecting only {objects} with {selection_type} > {resolution_limit:.2e} Ms")
    print(f" and within {selection_type} range {min_mass:.2e} < M/Ms < {max_mass:.2e}")

    # select mass range
    m_stars_all = h.get_properties_all_halos()["SubhaloMassType"][:, 4]
    m_tot_all = h.get_properties_all_halos()["SubhaloMass"]
    if selection_type == "stellar mass":
        mass_mask = (m_stars_all > min_mass) * (m_stars_all < max_mass)
    if selection_type == "total mass":
        mass_mask = (m_tot_all > min_mass) * (m_tot_all < max_mass)
    print(f" ... selected {sum(mass_mask)} subhalos in mass range")

    i = 0
    for isub in range(len(m_tot_all)):
        # Group properties
        mtot = m_tot_all[isub]
        ms = m_stars_all[isub]

        if selection_type == "stellar mass":
            mass = ms
        if selection_type == "total mass":
            mass = mtot

        if subhalo.properties["SubhaloFlag"] != 1:
            continue
        if mass < resolution_limit:
            continue
        if mass < min_mass:
            continue
        if mass > max_mass:
            continue
        # if central_flag not in primary_flag: continue

        # central subhalo properties
        subhalo = h[0]
        subid = isub
        ms = subhalo.properties["SubhaloMassType"][4]
        mtot = subhalo.properties["SubhaloMass"]
        rh = subhalo.properties["SubhaloHalfmassRadType"][4]
        v0 = subhalo.properties[catalog_fields[0]]
        v1 = subhalo.properties[catalog_fields[1]]

        # if debug and subid%1000==0: print('\nGalaxy:',i,' subif:',subid,' gid:',gid,'flag:', central_flag, 'mass:',np.log10(mass))

        # print galaxy info
        print(f"\n SubId:{subid}, Mstars={ms}, Rhalf={rh}, Mhalo={mh}")

        # load galaxy particles
        print("\n loading halo particles...")
        if component == "stars":
            particles = subhalo.stars  # all fields
        elif component == "gas":
            particles = subhalo.gas
        elif component == "dm":
            particles = subhalo.dm
        # print number of particles in the galaxy
        print(f"\n number of {component} particles: {len(particles)}")
        totmass_temp = np.sum(particles["mass"])
        print(f"\n total particle mass: {totmass_temp}")

        # center the coordinates/velocities and masses
        particles["pos"] = particles["pos"] - subhalo.properties["SubhaloPos"]
        particles["vel"] = particles["vel"] - subhalo.properties["SubhaloVel"]

        # rotate galaxy
        particles = rotate_galaxy(particles, orientation, spin_aperture)

        # create and save image
        if output_type == "2D projection":
            print(" creating image...")
            create_image(
                particles,
                field,
                operation,
                fov,
                image_depth,
                image_scale,
                image_size,
                smoothing,
                subid,
                component,
                orientation,
                output_path,
                debug,
            )

        if output_type == "point cloud":
            print(" creating point cloud...")
            pointcloud = [particles["pos"], particles["vel"], particles[field]]

        sub_id.append(subid)
        # group_id.append(gid)
        m_stellar.append(ms)
        m_halo.append(mh)
        m_tot.append(mtot)
        r_half.append(rh)
        var0.append(v0)
        var1.append(v1)
        i += 1

        del subhalo

    print("\nCreating catalog...")
    catalog_props = ["SubID", "logMstar", "logMtot", "logMhalo", "Rhalf"]
    catalog_props = catalog_props + catalog_fields
    print(" properties:", catalog_props)
    array_list = [sub_id, np.log10(m_stellar), np.log10(m_tot), np.log10(m_halo), r_half, var0, var1]
    catalog = {}
    for prop_name, array in zip(catalog_props, array_list):
        catalog[str(prop_name)] = array

    print("... done")

    del subhalos

    return catalog


In [4]:
result = data_preprocess_local_pynbody(
    snapshot_path="/urz/gpuscratch/its/data/Illustris/TNG50-1/snapshot_099/snap_099",
    # halos_path="/urz/gpuscratch/its/data/Illustris/TNG50-1/groups_099/fof_subhalo_tab_099",
    objects="centrals",
    min_mass=5e10,
    max_mass=5.2e10,  # [Msun]
    component="stars",
    field="mass",
    fov="scaled",  # [kpc]
    image_depth=1.0,  #  1 particles per pixel (min. S/N=sqrt(depth))
    image_size=128,
    smoothing=0.0,  # [kpc]
    image_scale="log",
    orientation="original",
    output_path="./PEST/images_test_pynbody/",
)

Parameters:
 snapshot_path: /urz/gpuscratch/its/data/Illustris/TNG50-1/snapshot_099/snap_099
 halos_path: None
 objects: centrals
 selection_type: stellar mass
 min_mass: 5.00e+10 Ms, max_mass: 5.20e+10 Ms
 component: stars
 output_type: 2D projection
 field: mass
 operation: sum
 fov: scaled
 image_depth: 1.0 particles
 image_size: 128
 smoothing: 0.0 kpc
 channels: 1
 image_scale: log
 orientation: original
 spin_aperture: 15.0 kpc
 catalog_fields: ['SubhaloStarMetallicity', 'SubhaloSFR']
 resolution_limit: 1.00e+09 Ms
 output_path: ./PEST/images_test_pynbody/

loading simulation snapshot from /urz/gpuscratch/its/data/Illustris/TNG50-1/snapshot_099/snap_099...

 snapshot properties:
 {'a': np.float64(0.9999999999999998), 'omegaB0': np.float64(0.0486), 'omegaM0': np.float64(0.3089), 'omegaL0': np.float64(0.6911), 'boxsize': Unit("1.08e+26 cm a h**-1"), 'h': np.float64(0.6774), 'time': Unit("1.38e+01 Gyr"), 'Composition_vector_length': np.int32(0), 'Flag_Cooling': np.int32(1), 'Flag_Do

UnboundLocalError: cannot access local variable 'subhalo' where it is not associated with a value